<a href="https://colab.research.google.com/github/xXLukaENPXx/Algoritmos-2/blob/main/Evaluation_Protocol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# -*- coding: utf-8 -*-
"""
Protocolo de Evaluación para Rendimiento de Clasificación con Datos Basados en Pacientes
Implementación en Google Colab con validación LOPO.
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from collections import Counter
from tqdm import tqdm # Para mostrar una barra de progreso

# --- 1. Carga y Preparación de Datos ---
print("Cargando datos desde la URL...")
url = "https://raw.githubusercontent.com/Fernoguez/EEG/main/DM.csv"
try:
    df = pd.read_csv(url)
    print("Datos cargados exitosamente.")
except Exception as e:
    print(f"Error al cargar los datos: {e}")
    # En caso de error, se puede crear un DataFrame de ejemplo o detener la ejecución
    df = pd.DataFrame()

if not df.empty:
    # Identificar columnas de características y etiquetas según la estructura descrita
    # Asumimos que 'Grupo' es la etiqueta de clase (0/1) y 'Participante' es el ID del paciente.
    id_column = 'Participante'
    label_column = 'Grupo'

    # Columnas a excluir para el entrenamiento
    cols_to_exclude = [id_column, label_column, 'Minuto']
    feature_columns = [col for col in df.columns if col not in cols_to_exclude]

    X = df[feature_columns]
    y = df[label_column]

    # Obtener lista única de pacientes para la validación LOPO
    patient_ids = df[id_column].unique()
    n_patients = len(patient_ids)
    print(f"Número total de pacientes: {n_patients}")
    print(f"Número total de cortes (muestras): {len(df)}")
    print(f"Características utilizadas: {feature_columns}")

    # --- 2. Validación Leave-One-Patient-Out (LOPO) y Evaluación ---
    # Estructuras para almacenar resultados de cada iteración
    patient_accuracies = [] # Para la macro-precisión (Promedio de Precisión del Paciente)
    patient_majority_vote_results = [] # Para la precisión de clasificación a nivel de paciente
    total_correct_cuts = 0 # Para la micro-precisión (Rendimiento a Nivel de Corte)
    total_cuts = 0

    # Barra de progreso para visualizar el avance de LOPO
    pbar = tqdm(patient_ids, desc="Procesando pacientes con LOPO")

    for test_patient in pbar:
        # --- División LOPO ---
        # Pacientes para entrenamiento: todos menos el paciente actual
        train_patients = [p for p in patient_ids if p != test_patient]

        # Índices para entrenamiento y prueba
        train_indices = df[df[id_column].isin(train_patients)].index
        test_indices = df[df[id_column] == test_patient].index

        X_train, X_test = X.loc[train_indices], X.loc[test_indices]
        y_train, y_test = y.loc[train_indices], y.loc[test_indices]

        # --- Pipeline de Modelado con Imputación de Datos Faltantes ---
        # Se crea un pipeline para imputar NaNs (usando la media) y luego clasificar.
        # Esto asegura que el modelo pueda manejar columnas con valores faltantes como 'EMG'.
        model = Pipeline([
            ('imputer', SimpleImputer(strategy='mean')), # Rellena NaN con la media de la columna
            ('classifier', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1))
        ])

        # Entrenar el modelo
        model.fit(X_train, y_train)

        # Predecir en el conjunto de prueba (cortes del paciente actual)
        y_pred = model.predict(X_test)

        # --- 2.1 Precisión a Nivel de Paciente (Sección 2.1 del documento) ---
        correct_cuts_patient = np.sum(y_pred == y_test)
        total_cuts_patient = len(y_test)
        patient_accuracy = correct_cuts_patient / total_cuts_patient if total_cuts_patient > 0 else 0
        patient_accuracies.append(patient_accuracy)

        # --- 2.2 Voto Mayoritario para Predicción a Nivel de Paciente (Sección 2.2 del documento) ---
        # Contar ocurrencias de cada clase predicha para los cortes de este paciente
        vote_counts = Counter(y_pred)
        # Determinar la clase mayoritaria. En caso de empate, se elige la primera (max).
        majority_class = vote_counts.most_common(1)[0][0]
        # La etiqueta verdadera del paciente es única, tomamos la primera de y_test
        true_class = y_test.iloc[0]
        # Registrar si la predicción mayoritaria fue correcta
        patient_majority_vote_results.append(majority_class == true_class)

        # --- 3.2 Acumuladores para Micro Precisión (Sección 3.2 del documento) ---
        total_correct_cuts += correct_cuts_patient
        total_cuts += total_cuts_patient

        # Actualizar descripción de la barra de progreso con métricas actuales
        pbar.set_postfix({
            'Paciente': test_patient,
            'Prec. Paciente': f"{patient_accuracy:.2f}",
            'Voto Correcto': majority_class == true_class
        })

    # --- 3. Cálculo de Métricas Finales ---
    print("\n" + "="*50)
    print("RESULTADOS DE LA EVALUACIÓN SEGÚN EL PROTOCOLO")
    print("="*50)

    # Macro Precisión (Promedio de Precisión del Paciente) - Sección 3.1
    average_patient_accuracy = np.mean(patient_accuracies)
    print(f"\n3.1 Macro Precisión (Promedio de Precisión del Paciente): {average_patient_accuracy:.4f}")

    # Precisión de Clasificación a Nivel de Paciente (Voto Mayoritario) - Sección 2.2
    patient_level_classification_accuracy = np.mean(patient_majority_vote_results)
    print(f"2.2 Precisión de Clasificación a Nivel de Paciente (Voto Mayoritario): {patient_level_classification_accuracy:.4f}")

    # Micro Precisión (Rendimiento a Nivel de Corte) - Sección 3.2
    micro_accuracy = total_correct_cuts / total_cuts if total_cuts > 0 else 0
    print(f"3.2 Micro Precisión (Rendimiento a Nivel de Corte): {micro_accuracy:.4f}")

else:
    print("No se pudieron cargar los datos. Finalizando la ejecución.")

Cargando datos desde la URL...
Datos cargados exitosamente.
Número total de pacientes: 18
Número total de cortes (muestras): 900
Características utilizadas: ['C3', 'C4', 'CZ', 'EMG', 'F3', 'F4', 'F7', 'F8', 'FP1', 'FP2', 'FZ', 'LOG', 'O1', 'O2', 'P3', 'P4', 'PZ', 'ROG', 'T3', 'T4', 'T5', 'T6']


Procesando pacientes con LOPO: 100%|██████████| 18/18 [00:02<00:00,  6.63it/s, Paciente=RRM, Prec. Paciente=0.70, Voto Correcto=True]


RESULTADOS DE LA EVALUACIÓN SEGÚN EL PROTOCOLO

3.1 Macro Precisión (Promedio de Precisión del Paciente): 0.5122
2.2 Precisión de Clasificación a Nivel de Paciente (Voto Mayoritario): 0.3333
3.2 Micro Precisión (Rendimiento a Nivel de Corte): 0.5122
